In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
import requests

/Users/garvitagarwal/Documents/LANGCHAIN 2/myenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
from langchain_community.tools import DuckDuckGoSearchRun

# search_tool = DuckDuckGoSearchRun()

# result = search_tool.invoke("What is top news today in INDIA")


/var/folders/3s/ns8l4nq14cq62jr4rlgm2hv80000gn/T/ipykernel_16627/2745747830.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [3]:
# result

In [4]:
from dotenv import load_dotenv
load_dotenv()
model = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
model.invoke("HI")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a18e1e-6ae6-7191-af9e-3493d92c4b73-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 27, 'total_tokens': 29, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 18}})

In [5]:
from langchain_classic.agents import create_react_agent, AgentExecutor

In [6]:
# Step 2: Pull the ReAct prompt from Langsmith client 

from langsmith import Client

client = Client() # we get predefine prompts from this hub

prompt = client.pull_prompt(
    "hwchase17/react",
    dangerously_pull_public_prompt=True
)

  # pulls the standard ReAct agent prompt 
# here react means reasisoning + action 


In [7]:
# # Step 3: Create the ReAct agent manually with the pulled prompt
# agent = create_react_agent(
#     llm=model,
#     tools=[search_tool],
#     prompt=prompt
#     )

In [8]:
# Step 4: Wrap it with AgentExecutor
# agent think and plan the execution and agentExecutor executes it 
# agent_executor = AgentExecutor(
#     agent=agent,
#     tools=[search_tool],
#     verbose=True # agent jo soch raha hoga vo vhi dikhai dega 
# ) 

In [9]:
from ddgs import DDGS
from langchain_core.tools import Tool

def search_web(query):
    with DDGS() as ddgs:
        results = ddgs.text(
            query,
            max_results=5,
            backend="duckduckgo, bing, brave"
        )

    return "\n\n".join(
        [f"Title: {r['title']}\nContent: {r['body']}" for r in results]
    )

search_tool = Tool(
    name="web_search",
    func=search_web,
    description="Search the internet for factual and current information."
)

In [10]:
agent = create_react_agent(
    llm=model,
    tools=[search_tool],
    prompt=prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool],
    verbose=True
)

In [ ]:
response = agent_executor.invoke({
    "input": "Find the capital of Madhya Pradesh, then find its current weather condition"
})

print(response)



> Entering new AgentExecutor chain...
Action: web_search
Action Input: capital of Madhya PradeshTitle: Madhya Pradesh - Wikipedia
Content: Madhya Pradesh (/ ˌmɑːdjə prəˈdɛʃ /; [10] Hindi: [ˈmədʱjə pɾəˈdeːʃ] ⓘ; lit. 'Central Province') is a state in central India. Its capital is Bhopal. Other major cities include Indore, Gwalior, Jabalpur, Chhindwara and Sagar. Madhya Pradesh is the second largest Indian state by area and the fifth largest state by population with over 72 million residents. It borders the states ...

Title: Bhopal - Wikipedia
Content: Bhopal is the capital city of the Indian state of Madhya Pradesh and the administrative headquarters of both Bhopal district and Bhopal division. It is known as the City of Lakes, due to presence of various natural and artificial lakes near the city boundary, and has a strong economic base with many large and medium industries.

Title: Capital of Madhya Pradesh Bhopal Facts and Details - Vedantu
Content: The capital of Madhya Pradesh, Bh

DDGSException: No results found.